In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score


In [2]:
df_law = pd.read_csv("law_dataset/bar_pass_prediction.csv")
columns_to_keep = ["age", "decile1", "decile3", "fam_inc", "lsat", "ugpa", "gender", "race1", "fulltime", "bar"]
df_law = df_law[columns_to_keep]

#count of missing values in each column
missing_values = df_law.isnull().sum()

df_law.dropna(inplace=True)
# drop age since they don't make sense 
df_law.drop(columns=['age'], inplace=True)

In [3]:
# Encode categorical variables
df_law['gender'] = pd.factorize(df_law['gender'])[0] # 0 for female, 1 for male
df_law['race1'] = pd.factorize(df_law['race1'])[0] # 0 for white, 1 for hispanic, 2 for asian, 3 for black, 4 for other
df_law['bar'] = pd.factorize(df_law['bar'])[0] # 0 for passed first time, 1 for failed, 2 for passed second time

# Reset index after dropping rows
df_law.reset_index(drop=True, inplace=True)

In [4]:
# take together 0 and 2 as passed, and 1 as failed for bar column
df_law['bar'] = df_law['bar'].replace({2: 0})

In [5]:
# imagine we only take 0 as passed first try and all the rest (so second try also) as failed first try 
df_law['bar'] = df_law['bar'].replace({2: 1})
# this slightly changes the target distibution favouring the ML model performance

In [6]:
df_law.shape

(20427, 9)

In [7]:
# view target distribution
print(df_law['bar'].value_counts())

bar
0    19389
1     1038
Name: count, dtype: int64


In [8]:
# define protected attributes 
protected_attributes = ["gender", "race1"]

In [9]:
df_law

,decile1,decile3,fam_inc,lsat,ugpa,gender,race1,fulltime,bar
0,10.0,10.0,5.0,44.0,3.5,0,0,1.0,0
1,5.0,4.0,4.0,29.0,3.5,0,0,1.0,0
2,3.0,2.0,1.0,36.0,3.5,1,0,1.0,0
3,7.0,4.0,4.0,39.0,3.5,1,0,1.0,0
4,9.0,8.0,4.0,48.0,3.5,1,0,1.0,0
...,...,...,...,...,...,...,...,...,...
20422,3.0,1.0,2.0,26.5,1.8,1,3,1.0,1
20423,3.0,1.0,3.0,19.7,1.8,1,3,1.0,1
20424,7.0,8.0,3.0,36.0,1.8,1,3,2.0,0
20425,10.0,10.0,3.0,44.0,1.5,1,0,2.0,0


In [10]:
#Load Data and Split
random_state = 1234
law_train, law_test = train_test_split(df_law, test_size=0.2, random_state=random_state)

law_train.to_parquet("law_dataset/train_cleaned.parquet")
law_test.to_parquet("law_dataset/test_cleaned.parquet")

# Separate features and target
x_train = law_train.drop("bar", axis=1)
y_train = law_train["bar"]
x_test = law_test.drop("bar", axis=1)
y_test = law_test["bar"] 

In [11]:
# random forest model
law_model=RandomForestClassifier(random_state=random_state)
law_model.fit(x_train, y_train)
predictions=law_model.predict(x_test)
target_pred_proba = law_model.predict_proba(x_test)[:, 1]  # Get probability for class 1
# evaluate model based on accuracy and AUC metrics
accuracy = accuracy_score(y_test, predictions)
auc = roc_auc_score(y_test, target_pred_proba)
print("Prediction Accuracy:", accuracy)
print("AUC:", auc)

Prediction Accuracy: 0.9481155163974547
AUC: 0.824418594128347


In [12]:
with open('law_dataset/RF.pkl', 'wb') as f:
    pickle.dump(law_model, f)

In [13]:
df_law.head()

,decile1,decile3,fam_inc,lsat,ugpa,gender,race1,fulltime,bar
0,10.0,10.0,5.0,44.0,3.5,0,0,1.0,0
1,5.0,4.0,4.0,29.0,3.5,0,0,1.0,0
2,3.0,2.0,1.0,36.0,3.5,1,0,1.0,0
3,7.0,4.0,4.0,39.0,3.5,1,0,1.0,0
4,9.0,8.0,4.0,48.0,3.5,1,0,1.0,0


In [14]:
feature_desc = ['the decile to which the student belongs in the school given his grades in Year 1.', 
                'the decile to which the student belongs in the school given his grades in Year 3.',
                'the family income bracket of the student (from 1 to 5).',
                'the LSAT score of the student.',
                'the undergraduate GPA of the student.',
                'the gender of the student (0 = female, 1 = male).',
                'the racial background of the student (0 = white, 1 = hispanic, 2 = asian, 3 = black, 4 = other).',
                'whether the student will work full-time or part-time (0 = part-time, 1 = full-time).'
]

feature_desc_df = pd.DataFrame({
    "feature_name": list(x_test.columns),
    "feature_average": x_train.mean().to_list() ,
    "feature_desc": feature_desc,
})
     

dataset_description="The dataset contains information about students who took the bar exam, including their academic performance, family background, and demographic information. Each row is a student."
target_description="The target variable is whether the student passed the bar exam on the first try (0 = passed, 1 = failed)"
task_description="Predict whether a student will pass the bar exam on the first try given all the other features"

dataset_info={
 "dataset_description": dataset_description,
 "target_description": target_description,
 "task_description": task_description,
 "feature_description": feature_desc_df
 }


with open('law_dataset/dataset_info', 'wb') as f:
    pickle.dump(dataset_info, f)

In [15]:
feature_desc_df

,feature_name,feature_average,feature_desc
0,decile1,5.711768,the decile to which the student belongs in the...
1,decile3,5.534606,the decile to which the student belongs in the...
2,fam_inc,3.467107,the family income bracket of the student (from...
3,lsat,36.698078,the LSAT score of the student.
4,ugpa,3.212925,the undergraduate GPA of the student.
5,gender,0.563919,"the gender of the student (0 = female, 1 = male)."
6,race1,0.373233,the racial background of the student (0 = whit...
7,fulltime,1.073374,whether the student will work full-time or par...
